# Clase 4. Programación dinámica: iteración de políticas y de valores

**Duración estimada:** 90 minutos
**Capítulo del manual:** [05_programacion_dinamica.md](../manual/05_programacion_dinamica.md)

## De "puedo evaluarla" a "puedo encontrar la mejor"

En la clase anterior calculaste Vπ, la función de valor de una política π fija, mediante evaluación iterativa: aplicaste la ecuación de Bellman una y otra vez, barrido tras barrido, hasta que los valores dejaron de cambiar. Eso es útil, pero incompleto. Saber cuánto vale seguir una política concreta no te dice si esa política es buena, igual que tener un mapa de distancias desde tu oficina hasta cada punto de la ciudad no te dice si tu ruta habitual es la mejor posible.

Hoy das el paso que faltaba: pasas de "puedo evaluar una política" a "puedo encontrar la mejor política posible". Al conjunto de herramientas que vas a usar se le llama **programación dinámica**: una familia de algoritmos que calculan políticas óptimas cuando conoces el modelo completo del entorno —las probabilidades de transición P y la función de recompensa R—, sin necesidad de interactuar con él por ensayo y error.

### Objetivos de hoy

- Entender la mejora de políticas y por qué nunca empeora el resultado
- Programar iteración de políticas: alternar evaluación completa y mejora hasta converger
- Programar iteración de valores: fundir evaluación y mejora en un único barrido
- Comparar el coste de ambos métodos sobre el mismo GridWorld

> 🎙️ **Guion de clase:** retoma la cuadrícula que dibujaste la clase pasada con los valores de la política aleatoria ya escritos. Pregunta: "si en (0,0) pudierais elegir una única acción distinta a la que dice la política, y luego seguir igual, ¿la cambiaríais?". Esa pregunta es, literalmente, la mejora de políticas.

## 1. De evaluar a mejorar

Imagina que ya calculaste Vπ para una política π cualquiera, por ejemplo la política aleatoria de la clase anterior. Ahora, párate en cada casilla del GridWorld y hazte una pregunta distinta: "si en vez de seguir la política actual tomo la mejor acción posible en este único paso, y luego vuelvo a seguir π, ¿mejoro mi resultado?".

Comparar esa opción con lo que hacía la política original y quedarte siempre con la mejor acción se llama **mejora de políticas**: construir una nueva política π' que actúa de forma codiciosa (*greedy*) respecto a Vπ, es decir, que en cada estado elige la acción que maximiza recompensa inmediata más valor descontado del siguiente estado.

Esto casi nunca empeora las cosas. Si en un estado la acción codiciosa coincide con lo que ya hacías, no pierdes nada; si es distinta, mejoras al menos un poco. Por eso π' es siempre igual o mejor que π en todos los estados —el mismo principio que aplicas cuando revisas tu ruta habitual al trabajo con la información de tráfico de hoy: en el peor de los casos sigues igual, en el mejor encuentras un atajo real—.

## 2. Iteración de políticas: evaluar y mejorar, una y otra vez

Si mejorar una política una vez ya ayuda, ¿qué pasa si repites el ciclo completo? Evalúas la política actual hasta el final (calculas su Vπ exacta con evaluación iterativa), la mejoras de forma codiciosa, evalúas la política nueva, la vuelves a mejorar, y así sucesivamente. A este procedimiento se le llama **iteración de políticas** (*policy iteration*).

Piensa en dos flechas que se persiguen. La evaluación es una flecha hacia arriba: ajusta los valores hasta que encajen a la perfección con la política actual. La mejora es una flecha hacia abajo: en cuanto tienes esos valores exactos, cambias la política, así que los valores que acabas de calcular ya no describen bien a la nueva política. El proceso repite ese vaivén hasta que, en un ciclo, la política ya no cambia.

En ese momento, la acción codiciosa en cada estado coincide exactamente con lo que la política ya hacía, lo cual solo puede significar una cosa: se cumple la ecuación de optimalidad de Bellman, y por tanto la política actual ya es la política óptima π\*. Como el número de políticas deterministas posibles en un PDM (Proceso de Decisión de Markov) finito es limitado, y cada ciclo produce una política igual o mejor que la anterior, el algoritmo tiene garantizado llegar a π\* en un número finito de ciclos.

> 🎙️ **Guion de clase:** dibuja las dos flechas en la pizarra (evaluación hacia arriba, mejora hacia abajo) y remarca el criterio de parada: el bucle termina cuando la política ya no cambia entre un ciclo y el siguiente, no cuando los valores dejan de cambiar.

## 3. Iteración de valores: el atajo directo

Iteración de políticas funciona, pero tiene un punto flojo: en cada ciclo, evaluar la política a fondo —repetir la evaluación iterativa hasta que converja del todo— puede costar muchos barridos, y buena parte de esa precisión se va a tirar en cuanto mejores la política de todas formas. ¿Hace falta esperar a que la evaluación converja del todo antes de mejorar?

La respuesta es no, y lleva directamente a **iteración de valores** (*value iteration*): en vez de completar la evaluación, la truncas a un único barrido y mezclas evaluación y mejora en el mismo paso. En cada barrido, cada estado actualiza su valor tomando directamente el máximo entre sus acciones —en lugar de promediar según una política fija—, usando la ecuación de optimalidad de Bellman como regla de actualización. Cuando esos valores dejan de cambiar de forma apreciable, derivas la política óptima con una última mirada de un paso: en cada estado, eliges la acción que maximiza recompensa más valor descontado del siguiente estado.

Puedes verlo como un caso particular de la misma familia que iteración de políticas: es iteración de políticas con la fase de evaluación recortada a un solo barrido. Suele necesitar más barridos individuales que ciclos completos necesita iteración de políticas, pero cada barrido es mucho más barato, así que el coste total termina siendo menor en muchos problemas.

## 4. Sincronía, actualizaciones en el sitio y otras variantes

Todo lo anterior asume programación dinámica **síncrona**: en cada barrido actualizas todos los estados a la vez, usando siempre los valores del barrido anterior. Existe también la programación dinámica **asíncrona**, que actualiza los estados de uno en uno (o por subconjuntos), usando el valor más reciente disponible en el momento, sin esperar a completar un barrido entero.

Una variante habitual son las actualizaciones **en el sitio** (*in-place*): en vez de mantener una copia "antigua" y una "nueva" de la función de valor, se actualiza directamente el mismo array, lo que suele acelerar la convergencia. Otras variantes priorizan qué estado actualizar primero según el tamaño del error de Bellman en cada casilla, o empiezan a actualizar cerca del estado terminal y avanzan hacia atrás. No profundices aquí: quédate solo con la idea de que, en problemas grandes, no hace falta barrer la cuadrícula entera de forma ordenada para que el método siga convergiendo.

> 🎙️ **Guion de clase:** no te detengas mucho en este punto, es un adelanto para problemas más grandes que los que vais a programar hoy. Una frase basta: "actualizar en orden ayuda a razonar sobre el algoritmo, pero no es un requisito para que converja".

## 5. Manos a la obra: iteración de políticas contra iteración de valores

Retomas el mismo GridWorld de la clase anterior: una cuadrícula de 4×4 (16 estados), con una única casilla terminal en la esquina inferior derecha, recompensa de −1 por cada paso que no llega a la meta y 0 al alcanzarla. Los movimientos son deterministas: si el agente intenta salir del tablero, se queda quieto.

Construye el código célula a célula: primero recuperas el entorno y la evaluación de políticas tal cual los dejaste la clase pasada, y después añades las dos piezas nuevas de hoy: mejora de políticas e iteración de valores. Ejecuta cada celda antes de pasar a la siguiente.

> 🎙️ **Guion de clase:** antes de teclear nada, pregunta "¿alguien recuerda las cuatro piezas de la clase GridWorld?". Deja que el grupo las nombre (estados, acciones, estado_siguiente, recompensa) antes de recuperar el código.

In [ ]:
# Solo necesitas numpy, ya viene preinstalado en Google Colab
import numpy as np

### El GridWorld, la política aleatoria y la evaluación: recordatorio

Estas tres celdas son idénticas a las que construiste en la Clase 3: la clase `GridWorld`, la `politica_aleatoria` y la `evaluacion_iterativa_politica`. Las recuperas tal cual porque las vas a necesitar dentro de un momento, dentro del bucle de iteración de políticas.

In [ ]:
ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

# Nueva de hoy: un símbolo por acción, para imprimir la política como flechas
FLECHAS = {"arriba": "^", "abajo": "v", "izquierda": "<", "derecha": ">"}


class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta).
    Misma clase de la Clase 3, sin cambios: la reutilizas tal cual.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None, recompensa_paso=-1.0):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)

        self.estados = [(f, c) for f in range(n_filas) for c in range(n_columnas)]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        if self.es_terminal(estado):
            return estado
        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        return (nueva_fila, nueva_columna)

    def recompensa(self, estado, accion, estado_siguiente):
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso

# 🎙️ Guion: si alguien llega nuevo hoy, esta celda no tiene nada distinto a
# la Clase 3. No te pares a explicarla de nuevo, solo confirma que corre.

In [ ]:
def politica_aleatoria(mundo):
    """Política de partida: en cada estado, las cuatro acciones son
    igual de probables. Es el punto de arranque de iteración de políticas."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }

In [ ]:
def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Aplica la ecuación de Bellman de expectación barrido tras barrido,
    hasta que Vpi deja de cambiar. Devuelve V y el número de barridos usados."""
    V = {estado: 0.0 for estado in mundo.estados}
    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)
        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue
            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])
            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))
        V = V_nuevo
        if delta_maximo < umbral:
            break
    return V, iteracion

# 🎙️ Guion: pregunta qué devuelve esta función (V y el número de barridos)
# antes de continuar. Es la pieza que vas a llamar dentro del bucle nuevo de hoy.

### Pieza nueva 1: mejora de políticas

Aquí empieza lo nuevo de hoy. Dada una Vπ ya calculada, `mejora_politica` recorre cada estado y elige, de forma codiciosa, la acción que maximiza recompensa inmediata más valor descontado del siguiente estado. `valores_q` es exactamente Q(estado, acción) de un solo paso.

> 🎙️ **Guion de clase:** señala línea a línea la correspondencia con la fórmula: `valores_q[accion] = r + gamma * V[estado_sig]` es `R(s,a,s') + γ·Vπ(s')`, y `max(valores_q, key=valores_q.get)` es el "elige la mejor acción" de la mejora de políticas.

In [ ]:
def mejora_politica(mundo, V, gamma=0.9):
    """Construye una política determinista, codiciosa respecto a V: en cada
    estado, elige la acción que maximiza recompensa + gamma * V(siguiente)."""
    nueva_politica = {}
    for estado in mundo.estados:
        if mundo.es_terminal(estado):
            nueva_politica[estado] = {a: 0.0 for a in mundo.acciones}
            continue
        valores_q = {}
        for accion in mundo.acciones:
            estado_sig = mundo.estado_siguiente(estado, accion)
            r = mundo.recompensa(estado, accion, estado_sig)
            valores_q[accion] = r + gamma * V[estado_sig]
        mejor_accion = max(valores_q, key=valores_q.get)
        nueva_politica[estado] = {a: (1.0 if a == mejor_accion else 0.0) for a in mundo.acciones}
    return nueva_politica

### Pieza nueva 2: iteración de políticas

Junta evaluación y mejora en un bucle: evalúa la política actual hasta el final, mejórala, y repite hasta que la política deje de cambiar entre un ciclo y el siguiente. Ese es exactamente el criterio de parada: no comparas valores, comparas políticas.

In [ ]:
def iteracion_politicas(mundo, gamma=0.9, umbral=1e-4):
    """Alterna evaluación completa y mejora hasta que la política deja de cambiar."""
    politica = politica_aleatoria(mundo)
    ciclos_evaluacion_mejora = 0
    barridos_totales = 0
    while True:
        ciclos_evaluacion_mejora += 1
        V, barridos = evaluacion_iterativa_politica(mundo, politica, gamma, umbral)
        barridos_totales += barridos
        nueva_politica = mejora_politica(mundo, V, gamma)
        if nueva_politica == politica:  # la política ya no cambia: es óptima
            break
        politica = nueva_politica
    return politica, V, ciclos_evaluacion_mejora, barridos_totales

# 🎙️ Guion: dibuja otra vez las dos flechas que se persiguen (evaluación
# arriba, mejora abajo) señalando qué línea del código es cada una.

### Una función auxiliar: leer la política como un mapa de flechas

Antes de ejecutar, añades una función que imprime la mejor acción de cada estado como una flecha, para leer el resultado de un vistazo en vez de un diccionario plano.

In [ ]:
def imprimir_politica(mundo, politica):
    """Muestra la política como una cuadrícula de flechas, para leerla de un vistazo."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            estado = (f, c)
            if mundo.es_terminal(estado):
                fila_textos.append(" G ")
            else:
                mejor_accion = max(politica[estado], key=politica[estado].get)
                fila_textos.append(f" {FLECHAS[mejor_accion]} ")
        print("|".join(fila_textos))

### Ejecuta iteración de políticas

Crea el GridWorld y ejecuta iteración de políticas. Fíjate en dos números: cuántos ciclos completos de evaluación y mejora hacen falta, y cuántos barridos de evaluación se acumulan en total —esa segunda cifra es el coste real del método—.

> 🎙️ **Guion de clase:** antes de ejecutar, pregunta "¿hacia dónde creéis que va a apuntar la flecha en cada casilla?". Confirma la intuición: el camino más corto hacia la meta, combinando "abajo" y "derecha" según la posición.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
gamma = 0.9

politica_pi, V_pi, ciclos, barridos_pi = iteracion_politicas(mundo, gamma=gamma)

print(f"Ciclos evaluación+mejora: {ciclos}")
print(f"Barridos de evaluación acumulados: {barridos_pi}")
imprimir_politica(mundo, politica_pi)

### Pieza nueva 3: iteración de valores

Ahora la alternativa: en vez de evaluar hasta el final en cada ciclo, actualizas V con el máximo sobre acciones en cada barrido —la ecuación de optimalidad de Bellman—, y solo al final derivas la política con `mejora_politica`. Compara esta función con `evaluacion_iterativa_politica`: la única diferencia real es `max(valores_q)` en vez de un promedio ponderado por la política. Ese único cambio convierte evaluación en optimización.

> 🎙️ **Guion de clase:** proyecta las dos funciones lado a lado (o alterna entre celdas) y pide al grupo que señale la línea exacta que cambia. Es el momento de mayor densidad conceptual de la clase.

In [ ]:
def iteracion_valores(mundo, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Funde evaluación y mejora: cada barrido actualiza V con el máximo
    sobre acciones (ecuación de optimalidad de Bellman), no con un promedio
    según una política fija."""
    V = {estado: 0.0 for estado in mundo.estados}
    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)
        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue
            valores_q = []
            for accion in mundo.acciones:
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valores_q.append(r + gamma * V[estado_sig])
            V_nuevo[estado] = max(valores_q)
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))
        V = V_nuevo
        if delta_maximo < umbral:
            break
    politica_final = mejora_politica(mundo, V, gamma)  # una última mirada de un paso
    return politica_final, V, iteracion

### Ejecuta iteración de valores y compara

Ejecuta iteración de valores sobre el mismo GridWorld y compara el resultado con iteración de políticas: la política óptima y la V\* finales, y el coste de cada método.

In [ ]:
politica_vi, V_vi, barridos_vi = iteracion_valores(mundo, gamma=gamma)

print(f"Barridos hasta converger: {barridos_vi}")
imprimir_politica(mundo, politica_vi)

In [ ]:
diferencia_maxima = max(abs(V_pi[s] - V_vi[s]) for s in mundo.estados)

print(f"Diferencia máxima entre las dos V*: {diferencia_maxima:.6f}")
print("\nResumen del coste de cada método:")
print(f"  Iteración de políticas: {ciclos} ciclos, {barridos_pi} barridos totales")
print(f"  Iteración de valores:   {barridos_vi} barridos totales")

### Cómo leer el resultado

Con γ (gamma) = 0,9, ambos métodos llegan exactamente a la misma política óptima: desde cualquier casilla, la flecha apunta siempre hacia la meta por el camino más corto. La diferencia está en el esfuerzo: iteración de políticas necesita solo tres ciclos completos de evaluación y mejora, pero cada evaluación se ejecuta hasta el final, así que en total acumula 90 barridos. Iteración de valores llega al mismo resultado en apenas siete barridos, porque nunca espera a que la evaluación converja del todo antes de mejorar.

Este patrón es habitual: iteración de políticas suele necesitar menos ciclos "externos", pero cada uno cuesta más; iteración de valores hace muchos barridos baratos en vez de pocos barridos caros.

> 🎙️ **Guion de clase:** cierra esta parte con la frase clave: "menos ciclos caros, o más barridos baratos: dos caminos al mismo sitio". Pregunta al grupo cuál de los dos usarían si el GridWorld tuviera un millón de casillas en vez de dieciséis.

## Puntos clave

- La mejora de políticas construye una política nueva actuando de forma codiciosa (greedy) respecto a Vπ; nunca empeora la política original
- Iteración de políticas alterna evaluación completa y mejora hasta que la política deja de cambiar; en ese punto se cumple la ecuación de optimalidad de Bellman y la política es óptima
- Iteración de valores funde evaluación y mejora en un único barrido por iteración, usando el máximo sobre acciones en vez de un promedio según una política fija
- Ambos métodos requieren conocer el modelo completo del entorno (P y R); por eso pertenecen a la programación dinámica, no al aprendizaje por ensayo y error
- La programación dinámica asíncrona (actualizaciones en el sitio, priorización por error de Bellman, actualización desde el estado terminal hacia atrás) permite escalar la idea a problemas más grandes sin perder la garantía de convergencia

## Cierre y próximos pasos

Ya sabes evaluar una política y, con las dos herramientas de hoy, mejorarla hasta encontrar la política óptima π\*. Todo esto asumiendo que conocías el modelo completo del entorno: las probabilidades de transición P y la función de recompensa R.

**Antes de la próxima clase:**
- **Abre** el notebook de práctica `clase04_practica_programacion_dinamica.ipynb` y completa los dos ejercicios: programarás tú misma o tú mismo iteración de políticas e iteración de valores desde cero, sobre un GridWorld con un obstáculo añadido
- **Experimenta** cambiando γ a 0,5 y a 0,99 en el GridWorld de hoy: anota si cambia el número de barridos que necesita cada método para converger
- **Repasa** el capítulo 5 del manual si quieres profundizar en la comparación entre ambos métodos

En el próximo bloque del curso dejas atrás la suposición de conocer el modelo del entorno, y aprendes a evaluar y mejorar políticas solo a partir de la experiencia: Monte Carlo y diferencias temporales.

> 🎙️ **Guion de clase:** cierra recordando que hoy resolvisteis el problema "con el mapa completo en la mano". La gran pregunta que abre el próximo bloque es: ¿y si no tienes el mapa?